# Arrays em C

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx compila e roda a célula no seu computador. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Declarar e preencher arrays de tamanho fixo, inclusive 2D, e passá-los para funções junto com o tamanho.
- Explicar por que o C nunca confere um índice, e o que faz um acesso fora dos limites.
- Escolher onde um array grande fica: um array local na pilha, ou um static ou global.

**Neste notebook**, uma ideia de cada vez (rode cada célula antes de seguir):
1. Declarando um array
2. Sem .length: você guarda o tamanho
3. Na memória: lado a lado
4. Passando um array para uma função
5. O C nunca confere o índice
6. Arrays 2D
7. Arrays grandes e a stack

Os **exercícios** deste notebook terminam com linhas `check(...)`: testes mínimos que imprimem ✓ quando o seu código está certo. Rode a próxima célula uma vez antes, para eles poderem usar o `check`.

In [ ]:
// shared: check_int e check_str imprimem ✓ ou ✗; os exercícios abaixo usam
void check_int(const char *what, long actual, long expected) {
    if (actual == expected) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: esperado %ld, veio %ld\n", what, expected, actual);
    }
}

void check_str(const char *what, const char *actual, const char *expected) {
    if (actual == NULL) {
        actual = "(NULL)";
    }
    if (strcmp(actual, expected) == 0) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: esperado \"%s\", veio \"%s\"\n", what, expected, actual);
    }
}

## 1. Declarando um array

`int temps[7];` cria 7 ints lado a lado na memória. Você pode dar os primeiros valores entre chaves: `int temps[7] = {21, 23, 19};` e **o resto começa em 0**. Com a lista completa, dá para omitir o tamanho: `int primes[] = {2, 3, 5, 7};`. Um array local sem valores iniciais guarda **lixo** até você preenchê-lo.

In [ ]:
int temps[7] = {21, 23, 19};
for (int i = 0; i < 7; i++) {
    printf("%d ", temps[i]);
}
printf("\n");
int primes[] = {2, 3, 5, 7};
primes[0] = 11;
printf("primeiro: %d, último: %d\n", primes[0], primes[3]);

🧪 **Experimente:** escreva `int zeros[5] = {0};` e imprima: um jeito comum de começar um array em 0.

## 2. Sem .length: você guarda o tamanho

Um array em C **não sabe o próprio tamanho**. Guarde-o você mesmo, normalmente numa constante: `#define DAYS 7` faz o compilador trocar `DAYS` por `7` em todo lugar. Onde o array é declarado, `sizeof temps / sizeof temps[0]` também calcula: o total de bytes dividido pelos bytes de um elemento.

In [ ]:
#include <stdio.h>
#define DAYS 7

int main(void) {
    int temps[DAYS] = {21, 23, 19, 25, 22, 20, 24};
    int n = sizeof temps / sizeof temps[0];
    printf("%zu bytes no total, %d elementos\n", sizeof temps, n);
    int hot = 0;
    for (int i = 0; i < DAYS; i++) {
        if (temps[i] > 22) {
            hot++;
        }
    }
    printf("%d dias quentes\n", hot);
    return 0;
}

🧪 **Experimente:** mude `DAYS` para 9 e rode: os dois dias a mais começam em 0, e o laço acompanha a constante. (Com 5, os sete valores não caberiam mais.)

## 3. Na memória: lado a lado

Os elementos ficam **um ao lado do outro** na memória, cada um ocupando o `sizeof` do tipo dele. É assim que `a[i]` é encontrado tão rápido: o endereço de `a[0]` mais `i` vezes o tamanho de um elemento. Os slides mostram isso, e o que acontece um depois do fim.

#### 🎞️ Um array na memória

![int a[4] guarda quatro ints em sequência nos endereços 1000, 1004, 1008 e 1012; a[2] fica em 1008, e a[4] lê além do fim sem nenhuma conferência](array-address-steps.pt-br.svg)

In [ ]:
int a[4] = {10, 20, 30, 40};
for (int i = 0; i < 4; i++) {
    long offset = (char *) &a[i] - (char *) &a[0];
    printf("a[%d] = %d, %ld bytes depois de a[0]\n", i, a[i], offset);
}

🧪 **Experimente:** transforme num array de `double`, e imprima `a[i]` com `%g` no lugar de `%d`: qual a distância entre os elementos agora? E com `char` (impresso com `%d`)?

## 4. Passando um array para uma função

Uma função recebe **o próprio array**, não uma cópia, então ela pode mudar os elementos de quem chamou. Ela não recebe o tamanho, então passe-o como outro parâmetro. Um `const` no parâmetro promete que a função só lê o array, e o compilador cobra essa promessa.

In [ ]:
#include <stdio.h>

void double_all(int values[], int n) {
    for (int i = 0; i < n; i++) {
        values[i] *= 2;         // muda o array de quem chamou
    }
}

void print_all(const int values[], int n) {
    for (int i = 0; i < n; i++) {
        printf("%d ", values[i]);
    }
    printf("\n");
}

int main(void) {
    int prices[] = {3, 5, 8};
    double_all(prices, 3);
    print_all(prices, 3);
    return 0;
}

🧪 **Experimente:** tente mudar `values[0]` dentro de `print_all` e leia o erro: o `const` proíbe.

### ✏️ Exercício: Some um array

Devolva a soma dos n primeiros valores. Uma função não consegue perguntar o tamanho de um array, então o n vem junto.

In [ ]:
int sum_array(const int a[], int n) {
    // TODO: percorra os n valores
    return 0;
}

int main(void) {
    int a[] = {3, 1, 4, 1, 5};
    check_int("cinco valores", sum_array(a, 5), 14);
    check_int("só os dois primeiros", sum_array(a, 2), 4);
    check_int("nenhum", sum_array(a, 0), 0);
    return 0;
}

<details>
<summary>Ver uma resposta</summary>

```c
int sum_array(const int a[], int n) {
    int sum = 0;
    for (int i = 0; i < n; i++) {
        sum += a[i];
    }
    return sum;
}

int main(void) {
    int a[] = {3, 1, 4, 1, 5};
    check_int("cinco valores", sum_array(a, 5), 14);
    check_int("só os dois primeiros", sum_array(a, 2), 4);
    check_int("nenhum", sum_array(a, 0), 0);
    return 0;
}
```

</details>

### ✏️ Exercício: Conte acima de um limite

Conte os valores maiores que `limit` entre os n primeiros.

In [ ]:
int count_above(const int a[], int n, int limit) {
    // TODO: compare cada valor com limit
    return 0;
}

int main(void) {
    int a[] = {3, 9, 4, 12, 5};
    check_int("acima de 4", count_above(a, 5, 4), 3);
    check_int("acima de 20", count_above(a, 5, 20), 0);
    check_int("igual não é acima", count_above(a, 1, 3), 0);
    return 0;
}

<details>
<summary>Ver uma resposta</summary>

```c
int count_above(const int a[], int n, int limit) {
    int count = 0;
    for (int i = 0; i < n; i++) {
        if (a[i] > limit) {
            count++;
        }
    }
    return count;
}

int main(void) {
    int a[] = {3, 9, 4, 12, 5};
    check_int("acima de 4", count_above(a, 5, 4), 3);
    check_int("acima de 20", count_above(a, 5, 20), 0);
    check_int("igual não é acima", count_above(a, 1, 3), 0);
    return 0;
}
```

</details>

## 5. O C nunca confere o índice

O Java para com uma exceção quando um índice está fora da faixa. O C **não confere**: `a[4]` num array de 4 elementos compila e lê ou escreve a memória que vier em seguida. Isso é **comportamento indefinido**: o programa pode quebrar, estragar outras variáveis ou parecer funcionar. O Sphinx compila com verificações de memória que pegam isso e apontam a linha, para você achar enquanto aprende.

In [ ]:
int a[4] = {10, 20, 30, 40};
int sum = 0;
for (int i = 0; i < 4; i++) {
    sum += a[i];
}
printf("soma = %d\n", sum);

🧪 **Experimente:** mude `i < 4` para `i <= 4` e rode: leia o relatório da verificação de memória, que mostra o acesso inválido e a linha dele.

## 6. Arrays 2D

`int grid[3][4];` são 3 linhas de 4 ints, guardadas linha após linha. `grid[r][c]` alcança uma célula, e dois laços aninhados percorrem todas. Ao passar uma para uma função, o número de **colunas** precisa estar no parâmetro: `void show(int g[][4], int rows)`.

In [ ]:
#include <stdio.h>

void show(int g[][4], int rows) {
    for (int r = 0; r < rows; r++) {
        for (int c = 0; c < 4; c++) {
            printf("%3d", g[r][c]);
        }
        printf("\n");
    }
}

int main(void) {
    int table[3][4];
    for (int r = 0; r < 3; r++) {
        for (int c = 0; c < 4; c++) {
            table[r][c] = (r + 1) * (c + 1);
        }
    }
    show(table, 3);
    return 0;
}

🧪 **Experimente:** acrescente a soma de cada linha no fim dela.

## 7. Arrays grandes e a stack

Um array local mora na **stack**, que só guarda alguns megabytes. `int big[10000000];` dentro de uma função pode derrubar o programa antes de ele imprimir qualquer coisa. Para um array grande, faça-o `static` (ou global), o que o põe numa área separada que dura o programa inteiro, ou peça memória do heap com `malloc` (numa unidade mais à frente).

In [ ]:
static int counts[1000000];     // 4 MB, fora da stack, começa em 0
for (int i = 0; i < 1000000; i++) {
    counts[i % 10]++;
}
printf("counts[3] = %d\n", counts[3]);

🧪 **Experimente:** repare que um array `static` começa em 0 mesmo sem `= {0}`. Conte com ele quantos números de 0 a 999.999 terminam em 7.

## Em resumo

Um array em C é um número fixo de elementos de um tipo, lado a lado na memória. Ele não tem `.length`: você guarda o tamanho, normalmente numa constante, e passa para toda função que recebe o array.

```c
#include <stdio.h>

#define DAYS 7

double average(const int values[], int n) {
    int sum = 0;
    for (int i = 0; i < n; i++) {
        sum += values[i];
    }
    return (double) sum / n;
}

int main(void) {
    int temps[DAYS] = {21, 23, 19};      // o resto começa em 0
    printf("%.2f\n", average(temps, DAYS));
    return 0;
}
```

Uma função recebe o próprio array, não uma cópia, então pode mudar os elementos (`const` promete que não vai). Um array local fica na **pilha**, que tem só alguns megabytes: faça um array muito grande `static` ou global.

**Cuidado:** o C nunca confere um índice. `temps[7]` compila, mas é comportamento indefinido: pode ler ou sobrescrever outros dados, travar ou parecer funcionar. As verificações de memória do Sphinx pegam isso e dizem a linha.

## Leituras

- [Arrays](https://beej.us/guide/bgc/html/split/arrays.html) (Beej's Guide to C Programming, chapter 6, 10 min): O tamanho com sizeof, inicializadores, "Out of Bounds!", arrays multidimensionais e a passagem de arrays para funções. As partes com ponteiros da seção 6.6 fazem sentido completo depois da unidade de ponteiros.
- [Lecture 2: Arrays](https://cs50.harvard.edu/x/2026/notes/2/#arrays) (CS50x (Harvard), lecture notes, 6 min): Opcional: como um array fica na memória, um elemento depois do outro, com figuras. Pule "Reading Levels" e "Debugging" no começo.

## Pratique

O notebook cobriu o básico. Agora aplique nos desafios da unidade, na barra lateral do Sphinx, onde o Submit roda todos os testes, inclusive os ocultos:

- **Histograma** · Fácil
- **Crivo de Eratóstenes** · Fácil
- **Buffer circular** · Médio
- **Jogo da Vida** · Médio
- **Fatorial gigante** · Difícil